In [ ]:
import pandas
import numpy as np
import sys



In [2]:
data = pandas.read_csv('zoo.csv')
train = data.sample(frac=0.7)
test = data.drop(train.index)    

In [ ]:
# This section successfully loads the counts of the train dataset into these dictionaries
# feature_counts = {feature_name: {class_name: {feature_value: count}}}
class_counts = {}
feature_counts = {}
for index, row in train.iterrows():
    target = row['class_type']
    if target not in class_counts:
        class_counts[target] = 1
    else:
        class_counts[target] += 1
    for featureIndex, featureValue in row.items():
        if featureIndex not in feature_counts:
            feature_counts[featureIndex] = {}
        if target not in feature_counts[featureIndex]:
            feature_counts[featureIndex][target] = {}
        if featureValue not in feature_counts[featureIndex][target]:
            feature_counts[featureIndex][target][featureValue] = 1
        else:
            feature_counts[featureIndex][target][featureValue] += 1
    

In [ ]:
# This section calculates the class prior probability along with the likelihoods of each feature value.
# featureLikelihood = { feature_name : {class_name : {feature_value: likelihood}}}
alpha = 0.01
numberOfInstances = len(train)
classPriorProbability = {target: (count / numberOfInstances) for target, count in class_counts.items()}
featureLikelihood = {}
for feature_name in feature_counts:
    for target in feature_counts[feature_name]:
        for feature_value in feature_counts[feature_name][target]:
            count = feature_counts[feature_name][target][feature_value]
            if feature_name not in featureLikelihood:
                featureLikelihood[feature_name] = {}
            if target not in featureLikelihood[feature_name]:
                featureLikelihood[feature_name][target] = {}
            featureLikelihood[feature_name][target][feature_value] = (count + alpha) / (class_counts[target] + alpha * train[feature_name].nunique())

In [5]:
# TESTING PHASE
# Note: you might encounter feature values that were not in the train set. Their likelihoods are ( 0 + alpha ) / ( class_counts[target] + alpha * data[feature_name].nunique() )

test['predicted'] = -1
test['probability'] = -0.5
test['correct?'] = 'unknown'
totalCorrect = 0
totalInstances = len(test)
for index, row in test.iterrows():
    max_log_prob = -np.inf
    best_class = None
    for target in classPriorProbability:
        posterior_prob = np.log2(classPriorProbability[target])
        for featureIndex, featureValue in row.items():
            if featureIndex == 'class_type' or featureIndex == 'predicted' or featureIndex == 'probability' or featureIndex == 'correct?':
                continue
            if featureValue not in featureLikelihood[featureIndex][target]:
                #Feature value wasn't encountered during training
                posterior_prob += np.log2(alpha / (class_counts[target] + alpha * train[feature_name].nunique()))
            else:
                posterior_prob += np.log2(featureLikelihood[featureIndex][target][featureValue])
        if posterior_prob > max_log_prob:
            max_log_prob = posterior_prob
            best_class = target
    test.at[index, 'predicted'] = best_class
    test.at[index, 'probability'] = np.power(2, max_log_prob)
    if best_class == row['class_type']:
        totalCorrect += 1
        test.at[index, 'correct?'] = 'CORRECT'
    else:
        test.at[index, 'correct?'] = 'wrong'
accuracy = totalCorrect / totalInstances

test.to_csv(sys.stdout, index=False)

animal_name,hair,feathers,eggs,milk,airborne,aquatic,predator,toothed,backbone,breathes,venomous,fins,legs,tail,domestic,catsize,class_type,predicted,probability,correct?
buffalo,1,0,0,1,0,0,0,1,1,1,0,0,4,1,0,1,1,1,1.6281134503319732e-05,CORRECT
cavy,1,0,0,1,0,0,0,1,1,1,0,0,4,0,1,0,1,1,1.6123749616879003e-07,CORRECT
cheetah,1,0,0,1,0,0,1,1,1,1,0,0,4,1,0,1,1,1,1.845050618930504e-05,CORRECT
chicken,0,1,1,0,1,0,0,0,1,1,0,0,2,1,1,0,2,2,3.1319473209074613e-06,CORRECT
clam,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,7,7,2.032967457331252e-06,CORRECT
crayfish,0,0,1,0,0,1,1,0,0,0,0,0,6,0,0,0,7,7,4.0558206487056264e-06,CORRECT
flea,0,0,1,0,0,0,0,0,0,1,0,0,6,0,0,0,6,6,2.608577526759406e-06,CORRECT
gnat,0,0,1,0,1,0,0,0,0,1,0,0,6,0,0,0,6,6,1.0356827606242717e-05,CORRECT
gull,0,1,1,0,1,1,1,0,1,1,0,0,2,1,0,0,2,2,4.899616422985156e-06,CORRECT
kiwi,0,1,1,0,0,0,1,0,1,1,0,0,2,1,0,0,2,2,4.899616422985156e-06,CORRECT
ladybird,0,0,1,0,1,0,1,0,0,1,0,0,6,0,0,0,6,6,2.046844165056029e-08,CORRECT
lion,1,0,0,1,0,0,1,1,1,1,0